# LEXWeave continuation training


In [ ]:
%pip install "ijson>=3.2,<4" "tokenizers>=0.20,<1" "numpy>=1.24,<3" "huggingface_hub>=0.26,<2" "requests>=2.31,<3"

## 1. Your settings
Upload foundation_model.zip, then edit FOUNDATION_PATH and WORK here. Reuse identical data settings in both notebooks. Default streaming mode reconstructs and excludes the old selection using the ZIP manifests. Both notebooks must use identical corpus settings. LEX continues exported weights; Bridge trains from scratch. SOURCE_WEIGHTS is an experimental starting mixture, not a validated optimum. More data requires sufficient training time; MAX_HOURS pauses this invocation without finishing the token-based LR schedule.

In [ ]:
from pathlib import Path
import os
# ---- Kaggle version ----
DATA_MODE = 'stream'             # Private HF streaming by default; 'local' is optional.
DATA_ROOT = None                  # Only needed for DATA_MODE='local'.
REPO = 'Exploration-Lab/CS-781-Capstone'
HF_REVISION = None                # Uses old pinned revision for reconstruction, otherwise pins current commit.

# Kaggle unzips uploaded zips into /kaggle/input/<dataset-name>/..., so '/kaggle/input/foundation_model.zip' never exists.
# Auto-detect the extracted foundation folder (contains model.pt) or a leftover .zip.
def _find_foundation():
    inp = Path('/kaggle/input')
    if not inp.exists(): return None
    for p in sorted(inp.rglob('model.pt')):
        if (p.parent/'model_config.json').exists() or (p.parent/'tokenizer.json').exists(): return str(p.parent)
    for p in sorted(inp.rglob('*.zip')): return str(p)
    return None
FOUNDATION_PATH = _find_foundation()   # or set by hand, e.g. '/kaggle/input/<your-dataset>/<folder with model.pt>'
if FOUNDATION_PATH is None:
    raise FileNotFoundError('No foundation model found under /kaggle/input. Add your foundation dataset via Add Input, '
                            'or set FOUNDATION_PATH to the folder that contains model.pt.')

RECONSTRUCT_OLD_SELECTION = True  # Replay old manifest and verify totals; stores hashes, not old raw text.
WORK = Path('/kaggle/working/cs781_sagemaker')   # /kaggle/working is the only persistent+writable place
CACHE = WORK / 'shared_stream_v2'
OLD_CACHE = None
TOKENIZER_PATH = None
EXCLUDE_OLD_TRAIN = True
OLD_CHECKPOINT = None           # Auto-set to imported model.pt; used only in continuation mode.
START_MODE = 'continue'        # LEX: 'continue' or 'scratch'; Bridge: 'scratch'. Fresh optimizer for continuation.
RUN_TAG = 'v1'
ARCH = 'lexweave'
MODEL_SIZE = '95m'
ABLATION = 'full'
SEED = 42
CONTEXT = 1024
SOURCE_CONTEXT = 768
TARGET_CONTEXT = 256
VOCAB = 16384
BPE_CHARS = 30_000_000
VAL_FRACTION = .01
CHAR_BUDGETS = {'train_acts':500_000_000,'train_high_courts':2_000_000_000,
                'train_supreme_court':1_250_000_000,'train_tribunals':1_250_000_000}
SOURCE_WEIGHTS = {'train_acts':.10,'train_high_courts':.40,'train_supreme_court':.25,'train_tribunals':.25}
SHARD_DOCS = 1000
DEV_PER_SOURCE = 32
MICRO_BATCH = 4                 # If benchmark OOM: lower to 2 or 1, raise ACCUM to preserve effective batch.
ACCUM = 8
GRAD_CHECKPOINT = True
# 1B tokens is ~days on a T4. Size this to your GPU hours; the benchmark cell prints estimated_training_hours,
# so adjust these two numbers so that estimate is about 6-8 h, then rerun. Do NOT change them after training starts.
TRAIN_TARGET_TOKENS = 80_000_000
WARMUP_TOKENS = 1_600_000       # ~2% of the budget
LR = 3e-4
MIN_LR = 3e-5
MAX_HOURS = 8.0                 # Per invocation (Kaggle sessions end at 12 h); rerun train cell to continue.
EVAL_EVERY = 250
EVAL_BATCHES = 20
SAVE_EVERY = 100

# HF token from Kaggle secrets (Add-ons -> Secrets -> HF_TOKEN), only needed for streaming the data
try:
    from kaggle_secrets import UserSecretsClient
    os.environ.setdefault('HF_TOKEN', UserSecretsClient().get_secret('HF_TOKEN'))
except Exception:
    print('No Kaggle secret HF_TOKEN (you will be prompted, or the cache must already exist).')

os.environ['TOKENIZERS_PARALLELISM']='true'
os.environ['RAYON_NUM_THREADS']=str(min(8,os.cpu_count() or 1))
assert START_MODE in ('scratch','continue')
assert ARCH=='lexweave' or START_MODE=='scratch'
assert 0 < TARGET_CONTEXT <= CONTEXT and SOURCE_CONTEXT > 0
assert all(v > 0 for v in SOURCE_WEIGHTS.values()) and 0 < VAL_FRACTION < 1
for p in (OLD_CACHE,TOKENIZER_PATH,OLD_CHECKPOINT):
    if p is not None and not Path(p).exists():raise FileNotFoundError(p)
WORK.mkdir(parents=True,exist_ok=True)
RUN=WORK/'runs'/f'{ARCH}_{MODEL_SIZE}_{ABLATION}_{RUN_TAG}'
RUN.mkdir(parents=True,exist_ok=True)
print('Data mode:',DATA_MODE,'Foundation:',FOUNDATION_PATH,'Shared cache:',CACHE,'Run:',RUN,sep='\n')

## 2. Streaming preprocessing definitions
Raw training files remain on Hugging Face in streaming mode. One SQLite database stores duplicate hashes, selected text, metadata and progress. Ensure disk room for the selected text, token shards and checkpoints. If a file is changed, use a fresh CACHE; sampled file fingerprints are change guards, not full cryptographic verification of raw files.

In [ ]:
import os, json, math, time, random, hashlib, sqlite3, re, gc, zipfile, shutil
from pathlib import Path
from collections import Counter, defaultdict
from contextlib import nullcontext
import numpy as np
import ijson
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders

SOURCES = ['train_acts','train_high_courts','train_supreme_court','train_tribunals']

def atomic_json(obj, path):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix+'.tmp')
    tmp.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding='utf-8')
    os.replace(tmp, path)

def sha(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for b in iter(lambda:f.read(8<<20), b''): h.update(b)
    return h.hexdigest()

def fingerprint(path):
    if isinstance(path,RemoteJSON):
        return {'name':path.name,'bytes':path.size,'repo':REPO,'path':path.path,'revision':path.revision}
    path = Path(path)
    with path.open('rb') as f:
        a=f.read(1<<20); f.seek(max(0,path.stat().st_size-(1<<20))); b=f.read()
    return {'name':path.name,'bytes':path.stat().st_size,'edge_sha':hashlib.sha256(a+b).hexdigest()}

def records(path, require_id=False):
    with open_binary(path) as f:
        if path.suffix == '.jsonl':
            for i,line in enumerate(f):
                if line.strip():
                    row=json.loads(line)
                    if require_id and (not isinstance(row,dict) or 'id' not in row):
                        raise ValueError('An explicit ID is required')
                    yield str(row.get('id',i)) if isinstance(row,dict) else str(i),row
            return
        first=f.peek(4096).lstrip()
        if first.startswith(b'['):
            for i,row in enumerate(ijson.items(f,'item')):
                if require_id and (not isinstance(row,dict) or 'id' not in row):
                    raise ValueError('Official array records must carry IDs')
                yield str(row.get('id',i)) if isinstance(row,dict) else str(i),row
        elif first.startswith(b'{'):
            for key,row in ijson.kvitems(f,''): yield str(key),row
        else: raise ValueError(f'Expected JSON object/array or .jsonl: {path}')

def text_of(row):
    if isinstance(row,str): return row
    if isinstance(row,dict):
        for k in ('text','document','content','case_text','judgment','prompt'):
            if isinstance(row.get(k),str): return row[k]
    raise ValueError('Unrecognized text schema; inspect the record rather than silently skipping it')

def clean(s):
    # Preserve paragraphs, spelling, dates, citations, tables and short orders.
    return s.replace('\r\n','\n').replace('\r','\n').replace('\x00','').strip()

def canonical(s): return ' '.join(clean(s).split())
def digest(s): return hashlib.sha256(canonical(s).encode()).hexdigest()
def uniform(key, salt):
    return int(hashlib.sha256((salt+key).encode()).hexdigest()[:16],16)/2**64

def prepare_data():
    CACHE.mkdir(parents=True, exist_ok=True)
    paths=resolve_sources()
    reconstructed=reconstruct_old_selection(paths)
    old_paths={}
    if OLD_CACHE is not None:
        for split in ('train','val'):
            p=OLD_CACHE/(split+'.jsonl')
            if p.exists(): old_paths[split]=p
    if EXCLUDE_OLD_TRAIN and 'train' not in old_paths and reconstructed is None:
        raise ValueError('Strict exclusion requires OLD_CACHE/train.jsonl. Token .bin files are insufficient.')
    if START_MODE=='continue' and 'val' not in old_paths and reconstructed is None:
        raise ValueError('Continuation requires old val.jsonl or verified manifest reconstruction to protect validation.')
    cfg=dict(version=4,files=[fingerprint(p) for p in paths],budgets=CHAR_BUDGETS,
             seed=SEED,val_fraction=VAL_FRACTION,exclude_old_train=EXCLUDE_OLD_TRAIN,
             old={k:sha(p) for k,p in old_paths.items()},
             reconstruction_sha=sha(CACHE/'old_selection_report.json') if reconstructed else None)
    manifest=CACHE/'data_config.json'
    if manifest.exists():
        assert json.loads(manifest.read_text())==cfg,'Data config changed: choose a new CACHE_DIR.'
    else: atomic_json(cfg,manifest)
    db=sqlite3.connect(CACHE/'corpus.sqlite')
    db.execute('PRAGMA journal_mode=WAL')
    db.execute('CREATE TABLE IF NOT EXISTS old (h TEXT PRIMARY KEY, split TEXT)')
    db.execute('CREATE TABLE IF NOT EXISTS seen (h TEXT PRIMARY KEY)')
    db.execute('CREATE TABLE IF NOT EXISTS docs (h TEXT PRIMARY KEY, source TEXT, id TEXT, split TEXT, text TEXT, meta TEXT)')
    db.execute('CREATE TABLE IF NOT EXISTS progress (source TEXT PRIMARY KEY, state TEXT)')
    def state(k):
        r=db.execute('SELECT state FROM progress WHERE source=?',(k,)).fetchone()
        return json.loads(r[0]) if r else None
    def save(k,v): db.execute('INSERT OR REPLACE INTO progress VALUES (?,?)',(k,json.dumps(v)))
    try:
        if state('_old') is None:
            if reconstructed is not None:
                old_db=sqlite3.connect(reconstructed)
                try:
                    db.executemany('INSERT INTO old VALUES (?,?)',old_db.execute('SELECT h,split FROM membership'))
                finally:old_db.close()
            for split,p in old_paths.items():
                for key,row in records(p):
                    h=digest(text_of(row))
                    found=db.execute('SELECT split FROM old WHERE h=?',(h,)).fetchone()
                    if found and found[0]!=split: raise ValueError('Old train/val overlap detected')
                    db.execute('INSERT OR IGNORE INTO old VALUES (?,?)',(h,split))
            save('_old',{'done':True});db.commit()
        for source,path in zip(SOURCES,paths):
            st=state(source) or {'index':0,'done':False,'counts':{}}
            if st['done']: print(source,'already complete');continue
            counts=Counter(st['counts']);skip=st['index']; start=time.monotonic();last=start
            probability=min(1.,CHAR_BUDGETS[source]/max(1,path.stat().st_size))
            print(source,'selection probability',round(probability,4),'resume record',skip,flush=True)
            # Resume reparses the prefix of monolithic JSON but does not redo its cleaning/hashing.
            index=skip
            for index,(key,row) in enumerate(records(path),1):
                if index<=skip:continue
                s=clean(text_of(row));counts['records']+=1;counts['chars_scanned']+=len(s)
                if not s:
                    counts['empty']+=1
                else:
                    h=digest(s)
                    fresh=db.execute('INSERT OR IGNORE INTO seen VALUES (?)',(h,)).rowcount
                    if not fresh:
                        counts['duplicates']+=1;counts['duplicate_chars']+=len(s)
                    else:
                        old=db.execute('SELECT split FROM old WHERE h=?',(h,)).fetchone()
                        old=old[0] if old else None
                        if old:counts['old_'+old+'_overlap']+=1
                        # Old validation always stays held out. Old training never becomes new validation.
                        split=('val' if old=='val' else 'train' if old=='train' else
                               'val' if uniform(h,'split'+str(SEED))<VAL_FRACTION else 'train')
                        chosen=uniform(h,'select'+str(SEED))<probability
                        if old=='conflict':
                            counts['ambiguous_old_split_excluded']+=1
                        elif old=='train' and EXCLUDE_OLD_TRAIN:
                            counts['old_train_excluded']+=1
                        elif chosen:
                            # Only obvious content-free attestations are removed; no length cutoff.
                            junk=len(s)<300 and re.fullmatch(
                                r'[\s\S]*I attest to the accuracy and integrity of this document[\s.]*',s,re.I)
                            if junk:counts['attestation_only']+=1
                            else:
                                meta={k:str(row[k]) for k in ('court','court_id','year','tribunal','title','date')
                                      if isinstance(row,dict) and k in row}
                                db.execute('INSERT INTO docs VALUES (?,?,?,?,?,?)',
                                           (h,source,key,split,s,json.dumps(meta)))
                                counts[split+'_docs']+=1;counts[split+'_chars']+=len(s)
                if index%2000==0:
                    save(source,{'index':index,'done':False,'counts':dict(counts)});db.commit()
                if time.monotonic()-last>20:
                    print(source,index,'records;',dict(counts),'elapsed min',round((time.monotonic()-start)/60,1),flush=True)
                    last=time.monotonic()
            save(source,{'index':index,'done':True,'counts':dict(counts)});db.commit()
        report={s:state(s) for s in SOURCES}
        report['overlap_claim']=('Exact canonical-text exclusion against old selection verified (supplied text or manifest reconstruction); near duplicates not excluded.'
                                 if EXCLUDE_OLD_TRAIN else 'Old training text may be reused. No claim of an unseen-only corpus.')
        atomic_json(report,CACHE/'data_report.json')
        print(json.dumps(report,indent=2))
    except BaseException:
        db.rollback()  # stats/data since the last committed boundary are replayed together
        print('Stopped. Committed progress is safe; rerun this cell to resume.',flush=True)
        raise
    finally:db.close()
    return paths

def prepare_tokenizer():
    dest=CACHE/'tokenizer.json'
    cfg={'data_config':sha(CACHE/'data_config.json'),'external':sha(TOKENIZER_PATH) if TOKENIZER_PATH else None,
         'vocab':VOCAB,'chars':BPE_CHARS,'version':1}
    meta=CACHE/'tokenizer_config.json'
    if meta.exists():
        assert json.loads(meta.read_text())==cfg,'Tokenizer config changed; use a new CACHE_DIR'
        return Tokenizer.from_file(str(dest))
    if TOKENIZER_PATH:
        tokenizer=Tokenizer.from_file(str(TOKENIZER_PATH))
    else:
        assert START_MODE=='scratch','Do not change tokenizer when continuing a checkpoint'
        tokenizer=Tokenizer(models.BPE(unk_token='<unk>'))
        tokenizer.pre_tokenizer=pre_tokenizers.ByteLevel(add_prefix_space=False)
        tokenizer.decoder=decoders.ByteLevel()
        trainer=trainers.BpeTrainer(vocab_size=VOCAB,min_frequency=2,
            special_tokens=['<pad>','<bos>','<eos>','<unk>'],initial_alphabet=pre_tokenizers.ByteLevel.alphabet())
        def pieces():
            db=sqlite3.connect(CACHE/'corpus.sqlite')
            try:
                # Equal character budgets across sources; hash-order avoids file-prefix selection.
                cursors=[iter(db.execute("SELECT text FROM docs WHERE source=? AND split='train' ORDER BY h",(s,))) for s in SOURCES]
                used=[0]*4;cap=BPE_CHARS//4
                while any(c is not None and used[i]<cap for i,c in enumerate(cursors)):
                    for i,c in enumerate(cursors):
                        if c is None or used[i]>=cap:continue
                        try:s=next(c)[0]
                        except StopIteration:cursors[i]=None;continue
                        s=s[:cap-used[i]];used[i]+=len(s)
                        for j in range(0,len(s),16000):yield s[j:j+16000]
            finally:db.close()
        tokenizer.train_from_iterator(pieces(),trainer=trainer)
    assert tokenizer.get_vocab_size()<=65535
    assert all(tokenizer.token_to_id(s) is not None for s in ['<pad>','<bos>','<eos>','<unk>'])
    if TOKENIZER_PATH:
        shutil.copyfile(TOKENIZER_PATH,str(dest)+'.tmp')  # preserve the original tokenizer checksum
    else:tokenizer.save(str(dest)+'.tmp')
    os.replace(str(dest)+'.tmp',dest)
    atomic_json(cfg,meta)
    return tokenizer

def pack_data(tok):
    # Each shard is atomic. Resume skips committed shards; no repeated full-corpus tokenization.
    pack=CACHE/'packed';pack.mkdir(exist_ok=True)
    cfg={'version':2,'tokenizer':sha(CACHE/'tokenizer.json'),'data':sha(CACHE/'data_config.json'),'shard_docs':SHARD_DOCS}
    cp=pack/'config.json'
    if cp.exists():assert json.loads(cp.read_text())==cfg
    else:atomic_json(cfg,cp)
    if (pack/'report.json').exists() and (CACHE/'continuation_dev.json').exists():
        report=json.loads((pack/'report.json').read_text())
        metas=list(pack.glob('train_*.json'))+list(pack.glob('val_*.json'))
        valid=len(metas)==sum(v['shards'] for v in report.values())
        for p in metas:
            m=json.loads(p.read_text())
            valid=valid and p.with_suffix('.bin').exists() and p.with_suffix('.idx').exists()
            if valid:
                valid=(p.with_suffix('.bin').stat().st_size==2*m['tokens'] and
                       p.with_suffix('.idx').stat().st_size==8*(m['docs']+1))
        if valid:
            print('Complete token cache reused without rereading selected text.')
            return pack
    db=sqlite3.connect(CACHE/'corpus.sqlite'); stats={};dev=[]
    try:
        for split in ('train','val'):
            for source in SOURCES:
                cur=db.execute('SELECT h,id,text FROM docs WHERE source=? AND split=? ORDER BY h',(source,split))
                shard=0;total=0;docs=0;dev_n=0
                while True:
                    rows=cur.fetchmany(SHARD_DOCS)
                    if not rows:break
                    stem=f'{split}_{source}_{shard:05d}';done=pack/(stem+'.json')
                    if not done.exists():
                        offsets=[0];ids_meta=[];temp=pack/(stem+'.bin.tmp')
                        with temp.open('wb') as f:
                            # Threaded Rust BPE batching; cap batch characters to bound RAM.
                            batch=[];chars=0
                            def flush(batch):
                                if not batch:return
                                for (h,key,s),enc in zip(batch,tok.encode_batch([r[2] for r in batch])):
                                    ids=enc.ids+[tok.token_to_id('<eos>')]
                                    np.asarray(ids,dtype=np.uint16).tofile(f)
                                    offsets.append(offsets[-1]+len(ids));ids_meta.append([h,key])
                            for r in rows:
                                batch.append(r);chars+=len(r[2])
                                if len(batch)>=32 or chars>=1_000_000:flush(batch);batch=[];chars=0
                            flush(batch)
                        os.replace(temp,pack/(stem+'.bin'))
                        with (pack/(stem+'.idx.tmp')).open('wb') as f:np.asarray(offsets,dtype=np.int64).tofile(f)
                        os.replace(pack/(stem+'.idx.tmp'),pack/(stem+'.idx'))
                        atomic_json({'tokens':offsets[-1],'docs':len(rows),'ids':ids_meta},done)
                        print(stem,offsets[-1],'tokens',flush=True)
                    m=json.loads(done.read_text());total+=m['tokens'];docs+=m['docs'];shard+=1
                    if split=='val' and dev_n<DEV_PER_SOURCE:
                        for h,key,s in rows:
                            # Limit splitting memory for exceptionally large documents.
                            words=s.split(None,641)
                            if len(words)>=640 and dev_n<DEV_PER_SOURCE:
                                dev.append({'id':source+':'+key,'source':source,'hash':h,
                                            'prompt':' '.join(words[:512]),'reference':' '.join(words[512:640])})
                                dev_n+=1
                stats[split+'_'+source]={'tokens':total,'docs':docs,'shards':shard}
    finally:db.close()
    atomic_json(stats,pack/'report.json');atomic_json(dev,CACHE/'continuation_dev.json')
    print(json.dumps(stats,indent=2));print('Held-out continuations:',len(dev))
    return pack

class Corpus:
    def __init__(self,root,split):
        self.data=[];self.offsets=[];self.groups={s:[] for s in SOURCES};self.mass=[]
        for p in sorted(Path(root).glob(split+'_*.bin')):
            x=np.memmap(p,dtype=np.uint16,mode='r');off=np.memmap(p.with_suffix('.idx'),dtype=np.int64,mode='r')
            source=next(s for s in SOURCES if p.name.startswith(split+'_'+s+'_'))
            self.groups[source].append(len(self.data));self.data.append(x);self.offsets.append(off);self.mass.append(len(x))
        self.sources=[s for s in SOURCES if self.groups[s]]
        if len(self.sources)!=4:raise ValueError('Every split must contain all four sources; increase data budgets if needed')
        self.prob=np.array([SOURCE_WEIGHTS[s] for s in self.sources],float);self.prob/=self.prob.sum()
    def sample(self,rng,max_len):
        s=rng.choice(self.sources,p=self.prob); inds=self.groups[s]
        weights=np.array([self.mass[i] for i in inds],float);weights/=weights.sum()
        k=int(rng.choice(inds,p=weights));off=self.offsets[k];x=self.data[k]
        # Token-proportional document sampling, but never cross a document boundary.
        pos=int(rng.integers(len(x)));j=int(np.searchsorted(off,pos,side='right')-1)
        lo,hi=int(off[j]),int(off[j+1]);n=min(max_len,hi-lo)
        start=int(rng.integers(lo,hi-n+1))
        return np.array(x[start:start+n],dtype=np.int64)

def batch(corpus,rng,arch,micro=None):
    b=MICRO_BATCH if micro is None else micro
    if arch=='lexweave':
        x=np.full((b,CONTEXT),PAD,np.int64);y=np.full((b,CONTEXT),-100,np.int64)
        for i in range(b):
            seq=corpus.sample(rng,CONTEXT+1)
            if len(seq)<2:seq=np.array([BOS,EOS])
            n=len(seq)-1;x[i,:n]=seq[:-1];y[i,:n]=seq[1:]
        return torch.as_tensor(x,device=DEVICE),torch.as_tensor(y,device=DEVICE),None
    src=np.full((b,SOURCE_CONTEXT),PAD,np.int64)
    x=np.full((b,TARGET_CONTEXT),PAD,np.int64);y=np.full_like(x,-100)
    for i in range(b):
        seq=corpus.sample(rng,SOURCE_CONTEXT+TARGET_CONTEXT)
        if len(seq)<2:seq=np.array([BOS,EOS])
        # Random prefix length. The encoder NEVER sees continuation target tokens.
        hi=min(SOURCE_CONTEXT,len(seq)-1);lo=max(1,min(128,hi))
        cut=int(rng.integers(lo,hi+1));target=seq[cut:cut+TARGET_CONTEXT]
        src[i,:cut]=seq[:cut];x[i,0]=BOS
        if len(target)>1:x[i,1:len(target)]=target[:-1]
        y[i,:len(target)]=target
    return (torch.as_tensor(x,device=DEVICE),torch.as_tensor(y,device=DEVICE),torch.as_tensor(src,device=DEVICE))

import io
import getpass
from dataclasses import dataclass
from types import SimpleNamespace
from contextlib import contextmanager

_HF_TOKEN = None

@dataclass(frozen=True)
class RemoteJSON:
    path: str
    size: int
    revision: str
    @property
    def name(self): return Path(self.path).name
    @property
    def suffix(self): return Path(self.path).suffix
    @property
    def stem(self): return Path(self.path).stem
    def stat(self): return SimpleNamespace(st_size=self.size)

class HTTPReader(io.RawIOBase):
    def __init__(self,raw):super().__init__();self.raw=raw
    def readable(self):return True
    def readinto(self,b):
        data=self.raw.read(len(b),decode_content=True);n=len(data);b[:n]=data;return n

def hf_token():
    global _HF_TOKEN
    if _HF_TOKEN:return _HF_TOKEN
    from huggingface_hub import get_token
    _HF_TOKEN=os.environ.get('HF_TOKEN') or get_token() or getpass.getpass('Hugging Face READ token (hidden): ').strip()
    if not _HF_TOKEN:raise ValueError('A read token with access to the private assignment is required.')
    return _HF_TOKEN

@contextmanager
def open_binary(path):
    if not isinstance(path,RemoteJSON):
        with Path(path).open('rb') as f:yield f
        return
    import requests
    from huggingface_hub import hf_hub_url
    url=hf_hub_url(REPO,path.path,repo_type='dataset',revision=path.revision)
    try:
        # requests strips Authorization on cross-host redirects; signed storage URLs need no token.
        with requests.get(url,headers={'Authorization':'Bearer '+hf_token()},stream=True,timeout=(30,180)) as r:
            if r.status_code in (401,403):raise RuntimeError('HF access denied. Check account access and token read scope.')
            if r.status_code!=200:raise RuntimeError(f'HF returned HTTP {r.status_code} for {path.name}.')
            with io.BufferedReader(HTTPReader(r.raw),buffer_size=65536) as f:yield f
    except requests.RequestException:
        raise RuntimeError(f'Network read failed for {path.name}. Rerun to resume committed progress; no raw file was cached.') from None


def initialize_foundation():
    global OLD_CACHE,TOKENIZER_PATH,OLD_CHECKPOINT,RAW_MANIFEST_PATH
    RAW_MANIFEST_PATH=None
    if FOUNDATION_PATH is None:
        if START_MODE=='continue':raise ValueError('Continuation needs FOUNDATION_PATH or an explicitly configured checkpoint bundle.')
        print('No foundation bundle: scratch training; no old-data exclusion claim.')
        return
    p=Path(FOUNDATION_PATH)
    if not p.exists():raise FileNotFoundError(f'Upload foundation_model.zip to {p}, or set FOUNDATION_PATH to its extracted folder.')
    if p.is_file():
        stamp=sha(p);dest=WORK/'foundation_import'/stamp[:16];dest.mkdir(parents=True,exist_ok=True)
        allowed={'model.pt','tokenizer.json','raw_manifest.json','packed_manifest.json','tokenizer_manifest.json',
                 'requested_config.json','environment.json','metrics.json','README.txt'}
        with zipfile.ZipFile(p) as z:
            for name in allowed:
                members=[m for m in z.infolist() if not m.is_dir() and Path(m.filename).name==name]
                if len(members)>1:raise ValueError(f'Ambiguous {name} in ZIP')
                if not members:continue
                target=dest/name;m=members[0]
                if not target.exists() or target.stat().st_size!=m.file_size:
                    tmp=target.with_suffix(target.suffix+'.tmp')
                    with z.open(m) as src,tmp.open('wb') as out:shutil.copyfileobj(src,out,1<<20)
                    os.replace(tmp,target)
        bundle=dest
    else:
        hits=list(p.rglob('tokenizer.json'))
        if len(hits)!=1:raise ValueError('Expected one tokenizer.json in foundation folder')
        bundle=hits[0].parent
    if not (bundle/'tokenizer.json').exists():raise FileNotFoundError('tokenizer.json missing from foundation bundle')
    TOKENIZER_PATH=bundle/'tokenizer.json'
    if (bundle/'model.pt').exists():OLD_CHECKPOINT=bundle/'model.pt'
    if (bundle/'raw_manifest.json').exists():RAW_MANIFEST_PATH=bundle/'raw_manifest.json'
    if START_MODE=='continue' and OLD_CHECKPOINT is None:raise FileNotFoundError('model.pt missing')
    if EXCLUDE_OLD_TRAIN and not RECONSTRUCT_OLD_SELECTION and OLD_CACHE is None:
        raise ValueError('Strict exclusion requires reconstruction or the original text cache')
    if RECONSTRUCT_OLD_SELECTION:
        if RAW_MANIFEST_PATH is None:raise FileNotFoundError('raw_manifest.json is required for reconstruction')
        tm=bundle/'tokenizer_manifest.json'
        if tm.exists():
            expected=json.loads(tm.read_text()).get('raw_sha')
            if expected and expected!=sha(RAW_MANIFEST_PATH):raise ValueError('Foundation manifest integrity mismatch')
    if START_MODE=='continue':
        ck=torch.load(OLD_CHECKPOINT,map_location='cpu',weights_only=False)
        if not isinstance(ck,dict) or not {'model_config','model'}<=set(ck):
            raise ValueError('Unexpected model.pt schema; need a weights/config checkpoint from the old export.')
        if ck['model_config'].get('arch')!='lexweave':raise ValueError('This is not the expected LEXWeave checkpoint')
        if MODEL_SIZE!='95m':raise ValueError('Continuing the old model requires MODEL_SIZE=95m. Larger models start from scratch.')
        expected=ck.get('tokenizer_sha',ck.get('train_config',{}).get('tokenizer_sha'))
        if expected!=sha(TOKENIZER_PATH):raise ValueError('model.pt and tokenizer.json do not have matching checksums')
        print('Exported checkpoint step:',ck.get('step'),'state tensor elements:',sum(v.numel() for v in ck['model'].values()))
        del ck;gc.collect()
    print('Foundation imported:',bundle)
    print('Tokenizer reused. Model weights', 'will be continued with a NEW optimizer.' if START_MODE=='continue' else 'are NOT loaded (scratch run).')
    print('No old train/val texts or optimizer state inferred merely from the ZIP filenames.')


def resolve_sources():
    CACHE.mkdir(parents=True,exist_ok=True)
    if DATA_MODE=='local':
        if DATA_ROOT is None:raise ValueError('Set DATA_ROOT for local mode')
        paths=[]
        for s in SOURCES:
            hits=list(DATA_ROOT.rglob(s+'.json'))
            if len(hits)!=1:raise ValueError(f'Expected exactly one {s}.json')
            paths.append(hits[0])
        return paths
    assert DATA_MODE=='stream'
    cache=CACHE/'hf_sources.json'
    old_revision=None
    if RECONSTRUCT_OLD_SELECTION and RAW_MANIFEST_PATH is not None:
        old_revision=json.loads(RAW_MANIFEST_PATH.read_text())['config'].get('revision')
        if not isinstance(old_revision,str) or not re.fullmatch(r'[0-9a-fA-F]{40}',old_revision):old_revision=None
    requested=HF_REVISION or old_revision
    if HF_REVISION and old_revision and HF_REVISION!=old_revision:
        raise ValueError('Reconstruction requires the old pinned revision; choose it for this experiment.')
    if cache.exists():
        info=json.loads(cache.read_text())
        assert info['repo']==REPO
        if requested and info['revision']!=requested:raise ValueError('Dataset revision changed: use a new CACHE')
    else:
        from huggingface_hub import HfApi
        try:remote=HfApi().dataset_info(REPO,revision=requested,token=hf_token(),files_metadata=True)
        except Exception:
            raise RuntimeError('Cannot list the private HF dataset. Check token read access and SageMaker internet access.') from None
        sizes={s.rfilename:s.size for s in remote.siblings}
        def pick(name,split):
            matches=[p for p in sizes if p.startswith('Phase_1_Pretraining/'+split+'/') and Path(p).name==name]
            if len(matches)!=1:raise ValueError(f'Expected one official {split}/{name}; got {matches}')
            p=matches[0]
            if not isinstance(sizes[p],int) or sizes[p]<=0:raise ValueError(f'Missing byte size for {p}')
            return {'path':p,'size':sizes[p],'revision':remote.sha}
        info={'repo':REPO,'revision':remote.sha,'train':[pick(s+'.json','Train') for s in SOURCES],
              'test':pick('test.json','Test')}
        atomic_json(info,cache)
    print('Official dataset revision:',info['revision'])
    print('Streaming JSON: no snapshot_download, no full raw training files stored on disk.')
    return [RemoteJSON(**r) for r in info['train']]


def official_test_path():
    if DATA_MODE=='local':
        hits=list(DATA_ROOT.rglob('test.json'))
        if len(hits)!=1:raise ValueError(f'Expected one test.json; got {hits}')
        return hits[0]
    # Called only during submission generation, never during corpus construction.
    info=json.loads((CACHE/'hf_sources.json').read_text());remote=RemoteJSON(**info['test'])
    dest=CACHE/'official_test'/remote.revision/'test.json';dest.parent.mkdir(parents=True,exist_ok=True)
    if not dest.exists() or dest.stat().st_size!=remote.size:
        tmp=dest.with_suffix('.partial')
        try:
            with open_binary(remote) as src,tmp.open('wb') as out:shutil.copyfileobj(src,out,65536)
            if tmp.stat().st_size!=remote.size:raise IOError('Incomplete test download')
            os.replace(tmp,dest)
        finally:tmp.unlink(missing_ok=True)
    return dest


def verify_old_sources(paths,manifest):
    cfg=manifest['config']
    if cfg.get('version')!=1:raise ValueError('Unsupported old preprocessing version; cannot reconstruct safely')
    expected=cfg.get('files',[])
    if [r.get('name') for r in expected]!=[p.name for p in paths]:raise ValueError('Old source order differs; reconstruction stopped')
    for p,e in zip(paths,expected):
        if p.stat().st_size!=e['size']:raise ValueError(f'Old source size mismatch: {p.name}')
        if isinstance(p,RemoteJSON):
            if e.get('revision')==p.revision and e.get('repo')==REPO and e.get('path')==p.path:continue
            # A local-source manifest does not pin immutable remote bytes. Fail closed rather than guessing.
            raise ValueError('Old manifest does not pin these HF files. Use original train/val JSONL, or scratch mode without old exclusion.')
        current=fingerprint(p)
        if e.get('edge_sha256')!=current['edge_sha']:raise ValueError(f'Old source fingerprint mismatch: {p.name}')


def reconstruct_old_selection(paths):
    """Replay the original version-1 round-robin selection, NOT the optimizer's random windows.
    Produces hashes only; excludes the entire old selected corpus (conservative superset of actual training exposure).
    """
    if not RECONSTRUCT_OLD_SELECTION or RAW_MANIFEST_PATH is None:return None
    manifest=json.loads(RAW_MANIFEST_PATH.read_text());verify_old_sources(paths,manifest)
    signature={'version':1,'raw_manifest_sha':sha(RAW_MANIFEST_PATH)}
    out=CACHE/'old_selection.sqlite';report_path=CACHE/'old_selection_report.json'
    if report_path.exists() and out.exists():
        report=json.loads(report_path.read_text())
        assert report['config']==signature
        print('Verified old-selection reconstruction reused:',report['documents'])
        return out
    out.unlink(missing_ok=True)
    db=sqlite3.connect(out)
    db.execute('CREATE TABLE membership (h TEXT PRIMARY KEY, split TEXT)')
    db.execute('CREATE TABLE raw_seen (h TEXT PRIMARY KEY)')
    cfg=manifest['config'];limits={'train':cfg['train_chars'],'val':cfg['val_chars']}
    sizes={'train':0,'val':0};counts={'train':Counter(),'val':Counter()};duplicates=0;conflicts=0;kept=0
    streams=[(p.stem,iter(records(p))) for p in paths];all_streams=[s for _,s in streams]
    started=last=time.monotonic()
    print('Reconstructing old selected documents from the pinned official streams (~old corpus size, not the full corpus).')
    try:
        while streams and any(sizes[k]<limits[k] for k in sizes):
            remaining=[]
            for source,stream in streams:
                try:key,row=next(stream)
                except StopIteration:continue
                remaining.append((source,stream));s=clean(text_of(row))
                if len(s.split())<32:continue
                raw_h=hashlib.sha256(s.encode()).hexdigest()
                split_hash=hashlib.sha256((str(cfg['seed'])+raw_h).encode()).hexdigest()
                split='val' if int(split_hash[:16],16)/2**64<cfg['val_fraction'] else 'train'
                if sizes[split]>=limits[split]:continue
                if not db.execute('INSERT OR IGNORE INTO raw_seen VALUES (?)',(raw_h,)).rowcount:
                    duplicates+=1;continue
                h=digest(s);existing=db.execute('SELECT split FROM membership WHERE h=?',(h,)).fetchone()
                if existing and existing[0]!=split:
                    db.execute("UPDATE membership SET split='conflict' WHERE h=?",(h,));conflicts+=1
                else:db.execute('INSERT OR IGNORE INTO membership VALUES (?,?)',(h,split))
                sizes[split]+=len(s);counts[split][source]+=1;kept+=1
                if time.monotonic()-last>20:
                    print('Old reconstruction:',kept,'documents;',sizes,'chars;',round((time.monotonic()-started)/60,1),'min',flush=True)
                    last=time.monotonic()
            streams=remaining
        actual={k:dict(v) for k,v in counts.items()}
        if sizes!=manifest['chars'] or actual!=manifest['documents'] or duplicates!=manifest['duplicates_removed']:
            raise ValueError('Reconstructed old selection DOES NOT match manifest counts/characters/duplicates. No exclusion claim is valid.')
        db.commit()
        atomic_json({'config':signature,'documents':actual,'chars':sizes,'duplicates':duplicates,
                     'canonical_conflicts':conflicts,'verified':True,
                     'scope':'Entire old raw selected corpus; not a record of optimizer-sampled windows. No near-duplicate guarantee.'},report_path)
        print('Old selection matches manifest exactly:',sizes)
        return out
    except BaseException:
        db.rollback();print('Old reconstruction incomplete; rerun this stage. New-data progress is unaffected.');raise
    finally:
        for stream in all_streams:stream.close()
        db.close()


## 3. Build or resume selected corpus
Run this on CPU before paying for the training GPU. SageMaker must have outbound internet access and your HF token must have access to the private repo. Logs report counts and elapsed time every ~20 seconds. The full streamed JSON scan is unavoidable for sampling across the entire file; selected data still need local working storage. Runtime must be measured on your disk/CPU. You may interrupt and rerun.

In [ ]:
initialize_foundation()
paths=prepare_data()

## 4. Tokenizer, shards and overlap report
If TOKENIZER_PATH is None, train a byte-level BPE on a source-balanced subset of selected training text. To isolate data changes from the old run, supply its tokenizer instead. Actual corpus size is printed; do not label a run “more data” if the selected count is not larger. Source sampling is token-proportional within each source, with replacement, and training windows do not cross documents. Short documents are right-padded with ignored targets.

In [ ]:
tok=prepare_tokenizer()
PAD,BOS,EOS,UNK=[tok.token_to_id(s) for s in ['<pad>','<bos>','<eos>','<unk>']]
V=tok.get_vocab_size()
PACK=pack_data(tok)
TRAIN=Corpus(PACK,'train');VAL=Corpus(PACK,'val')
report=json.loads((PACK/'report.json').read_text())
unique_train_tokens=sum(v['tokens'] for k,v in report.items() if k.startswith('train_'))
print('Selected unique-document train tokens:',f'{unique_train_tokens:,}')
print('Previous cached baseline: 178,389,638 tokens. New/base ratio:',round(unique_train_tokens/178389638,2))
if unique_train_tokens<=178389638:
    print('WARNING: this is NOT a larger token corpus than the previous baseline. Increase budgets in a new CACHE.')
print('Training samples with replacement: this does not guarantee one visit per document.')
print('Expected target-token exposure / corpus tokens:',round(TRAIN_TARGET_TOKENS/unique_train_tokens,2))


## 5. Manual model implementation

In [ ]:
"""LEXWeave: a project-specific experimental causal LM, not a pretrained model.

Parallel token attention, strictly completed-chunk memory, and routed dilated
convolution, combined by a token-dependent gate. See notebook for hypotheses,
prior work, limitations, and controlled ablations. No claim of literature novelty.
"""
import math
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint


class RMSNorm(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        z = x.float()
        return (z * torch.rsqrt(z.square().mean(-1, keepdim=True) + 1e-6)).to(x.dtype) * self.weight


class Rotary(nn.Module):
    def __init__(self, head_dim):
        super().__init__()
        assert head_dim % 2 == 0
        inv = 1.0 / (10000 ** (torch.arange(0, head_dim, 2).float() / head_dim))
        self.register_buffer('inv', inv, persistent=False)

    def forward(self, x, positions):
        a = torch.outer(positions.float(), self.inv.float())
        c = a.cos().to(x.dtype)[None, None]
        s = a.sin().to(x.dtype)[None, None]
        u, v = x[..., ::2], x[..., 1::2]
        return torch.stack((u*c-v*s, u*s+v*c), -1).flatten(-2)


class Attention(nn.Module):
    """Exact causal token attention is retained for names, numbers and citations."""
    def __init__(self, dim, heads, kv_heads):
        super().__init__()
        assert dim % heads == 0 and heads % kv_heads == 0
        self.h, self.k, self.dh = heads, kv_heads, dim // heads
        self.q = nn.Linear(dim, dim, bias=False)
        self.kv = nn.Linear(dim, 2 * kv_heads * self.dh, bias=False)
        self.out = nn.Linear(dim, dim, bias=False)
        self.rope = Rotary(self.dh)

    def forward(self, x):
        b, t, d = x.shape
        q = self.q(x).view(b, t, self.h, self.dh).transpose(1, 2)
        k, v = self.kv(x).view(b, t, 2, self.k, self.dh).unbind(2)
        k, v = k.transpose(1, 2), v.transpose(1, 2)
        pos = torch.arange(t, device=x.device)
        q, k = self.rope(q, pos), self.rope(k, pos)
        if self.h != self.k:
            k = k.repeat_interleave(self.h // self.k, dim=1)
            v = v.repeat_interleave(self.h // self.k, dim=1)
        z = F.scaled_dot_product_attention(q, k, v, is_causal=True, dropout_p=0.)
        return self.out(z.transpose(1, 2).contiguous().view(b, t, d))


class ChunkMemory(nn.Module):
    """Learn a salience-weighted vector per chunk; read only COMPLETED past chunks.

    Memory is recomputed inside the current context, not carried between batches.
    A fixed zero slot makes the first chunk well-defined without all-masked rows.
    """
    def __init__(self, dim, mem_dim=128, heads=4, chunk=32, learned_pool=True):
        super().__init__()
        assert mem_dim % heads == 0
        self.dim, self.md, self.h = dim, mem_dim, heads
        self.dh, self.chunk = mem_dim // heads, chunk
        self.score = nn.Linear(dim, 1, bias=False) if learned_pool else None
        self.q = nn.Linear(dim, mem_dim, bias=False)
        self.kv = nn.Linear(dim, 2*mem_dim, bias=False)
        self.out = nn.Linear(mem_dim, dim, bias=False)
        self.rope = Rotary(self.dh)

    def pool(self, x):
        b, t, d = x.shape
        c = self.chunk
        n = (t+c-1)//c
        padded = F.pad(x, (0, 0, 0, n*c-t)).view(b, n, c, d)
        valid = torch.arange(n*c, device=x.device).view(1, n, c) < t
        if self.score is None:
            w = valid.float().expand(b, -1, -1)
            w = w / w.sum(-1, keepdim=True).clamp_min(1)
        else:
            logits = self.score(padded).squeeze(-1).float()
            logits = logits.masked_fill(~valid, -float('inf'))
            w = logits.softmax(-1)
        summaries = (padded * w.to(padded.dtype).unsqueeze(-1)).sum(-2)
        ends = (torch.arange(n, device=x.device)+1)*c - 1
        return summaries, ends

    def forward(self, x):
        b, t, _ = x.shape
        summary, ends = self.pool(x)
        n = summary.shape[1]
        q = self.q(x).view(b, t, self.h, self.dh).transpose(1, 2)
        k, v = self.kv(summary).view(b, n, 2, self.h, self.dh).unbind(2)
        k, v = k.transpose(1, 2), v.transpose(1, 2)
        pos = torch.arange(t, device=x.device)
        q, k = self.rope(q, pos), self.rope(k, ends)
        # Null slot: exactly zero, no learned information from any future token.
        k = torch.cat((k.new_zeros(b, self.h, 1, self.dh), k), dim=2)
        v = torch.cat((v.new_zeros(b, self.h, 1, self.dh), v), dim=2)
        allowed = ends[None, :] < pos[:, None]
        allowed = torch.cat((torch.ones(t, 1, dtype=torch.bool, device=x.device), allowed), -1)
        z = F.scaled_dot_product_attention(q, k, v, attn_mask=allowed[None, None], dropout_p=0.)
        return self.out(z.transpose(1, 2).contiguous().view(b, t, self.md))


class PhraseMixer(nn.Module):
    """Token-gated, left-padded depthwise convolutions at multiple spacings."""
    def __init__(self, dim, inner=128, dilations=(1, 4, 16)):
        super().__init__()
        self.dilations = tuple(dilations)
        self.up = nn.Linear(dim, inner, bias=False)
        self.conv = nn.ModuleList([nn.Conv1d(inner, inner, 3, dilation=r,
                                            groups=inner, bias=False) for r in self.dilations])
        self.scale_gate = nn.Linear(inner, len(self.dilations), bias=False) if len(self.dilations)>1 else None
        self.out = nn.Linear(inner, dim, bias=False)

    def forward(self, x):
        z = F.silu(self.up(x))
        gate = self.scale_gate(z).float().softmax(-1).to(z.dtype) if self.scale_gate is not None else None
        y = torch.zeros_like(z)
        for i, (conv, r) in enumerate(zip(self.conv, self.dilations)):
            branch = conv(F.pad(z.transpose(1, 2), (2*r, 0))).transpose(1, 2)
            y = y + (branch * gate[..., i:i+1] if gate is not None else branch)
        return self.out(F.silu(y))


class WeaveBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        d, a = cfg['dim'], cfg['ablation']
        self.n1, self.n2 = RMSNorm(d), RMSNorm(d)
        self.attn = Attention(d, cfg['heads'], cfg['kv_heads'])
        self.memory = None if a in ('no_memory', 'plain') else ChunkMemory(
            d, cfg['mem_dim'], cfg['mem_heads'], cfg['chunk'], learned_pool=a != 'mean_pool')
        self.phrase = None if a in ('no_conv', 'plain') else PhraseMixer(
            d, cfg['conv_dim'], (1,) if a == 'single_scale' else cfg['dilations'])
        self.active = [0] + ([1] if self.memory is not None else []) + ([2] if self.phrase is not None else [])
        self.router = nn.Linear(d, 3, bias=False) if a not in ('fixed_mix', 'plain') else None
        self.register_buffer('route_prior', torch.tensor([2., 0., 0.]), persistent=True)
        self.chunk = cfg['chunk']
        self.up = nn.Linear(d, 2*cfg['ff_dim'], bias=False)
        self.down = nn.Linear(cfg['ff_dim'], d, bias=False)
        self.record_routes = False
        self.last_routes = None

    def route_weights(self, x):
        b, t, _ = x.shape
        logits = self.route_prior[None, None].expand(b, t, 3)
        if self.router is not None: logits = logits + self.router(x).float()
        enabled = torch.zeros(t, 3, dtype=torch.bool, device=x.device)
        enabled[:, self.active] = True
        # Chunk memory carries no past summary during the first chunk.
        enabled[:, 1] &= torch.arange(t, device=x.device) >= self.chunk
        return logits.masked_fill(~enabled[None], -float('inf')).softmax(-1).to(x.dtype)

    def forward(self, x):
        z = self.n1(x)
        weights = self.route_weights(z)
        mixed = weights[..., 0:1] * self.attn(z)
        if self.memory is not None: mixed = mixed + weights[..., 1:2] * self.memory(z)
        if self.phrase is not None: mixed = mixed + weights[..., 2:3] * self.phrase(z)
        if self.record_routes and not self.training:
            self.last_routes = weights.detach().float().mean((0, 1)).cpu()
        x = x + mixed
        a, b = self.up(self.n2(x)).chunk(2, -1)
        return x + self.down(F.silu(a) * b)


class LegalLM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg, self.kind = dict(cfg), 'lexweave'
        self.grad_ckpt = cfg.get('grad_checkpoint', True)
        self.emb = nn.Embedding(cfg['vocab'], cfg['dim'])
        self.blocks = nn.ModuleList([WeaveBlock(cfg) for _ in range(cfg['layers'])])
        self.norm = RMSNorm(cfg['dim'])
        self.apply(self.init_weights)
        # Mild residual initialization keeps the extra paths from overwhelming input.
        residual_std = 0.02 / math.sqrt(2*cfg['layers'])
        for block in self.blocks:
            nn.init.normal_(block.attn.out.weight, std=residual_std)
            nn.init.normal_(block.down.weight, std=residual_std)
            if block.memory is not None:
                nn.init.normal_(block.memory.out.weight, std=residual_std)
                if block.memory.score is not None: nn.init.zeros_(block.memory.score.weight)
            if block.phrase is not None:
                nn.init.normal_(block.phrase.out.weight, std=residual_std)
                if block.phrase.scale_gate is not None: nn.init.zeros_(block.phrase.scale_gate.weight)
            if block.router is not None: nn.init.zeros_(block.router.weight)

    @staticmethod
    def init_weights(m):
        if isinstance(m, (nn.Linear, nn.Embedding, nn.Conv1d)):
            nn.init.normal_(m.weight, std=0.02)
            if getattr(m, 'bias', None) is not None: nn.init.zeros_(m.bias)

    def hidden(self, ids, src=None, memory=None):
        if ids.shape[1] > self.cfg['context']: raise ValueError('Input exceeds configured context.')
        x = self.emb(ids)
        for block in self.blocks:
            x = checkpoint(block, x, use_reentrant=False) if self.training and self.grad_ckpt else block(x)
        return self.norm(x)

    def forward(self, ids, src=None, targets=None, memory=None):
        x = self.hidden(ids)
        if targets is None: return F.linear(x[:, -1], self.emb.weight).float()
        def ce(h, y):
            logits = F.linear(h, self.emb.weight).float()
            return F.cross_entropy(logits.reshape(-1, self.cfg['vocab']), y.reshape(-1), reduction='sum')
        total = x.new_zeros((), dtype=torch.float32)
        for j in range(0, x.shape[1], 128):
            h, y = x[:, j:j+128], targets[:, j:j+128]
            total = total + (checkpoint(ce, h, y, use_reentrant=False) if self.training and self.grad_ckpt else ce(h, y))
        return total / targets.ne(-100).sum().clamp_min(1)




In [ ]:
def model_config():
    sizes={'95m':dict(dim=768,heads=12,kv_heads=4,layers=12,ff_dim=2048),
           '182m':dict(dim=1024,heads=16,kv_heads=4,layers=14,ff_dim=2736),
           '230m':dict(dim=1024,heads=16,kv_heads=4,layers=18,ff_dim=2736)}
    assert ABLATION in ('full','no_memory','no_conv','mean_pool','single_scale','fixed_mix','plain')
    return dict(arch='lexweave',ablation=ABLATION,
                vocab=V,context=CONTEXT,grad_checkpoint=GRAD_CHECKPOINT,mem_dim=128,mem_heads=4,
                conv_dim=128,chunk=32,dilations=(1,4,16),**sizes[MODEL_SIZE])
CFG=model_config()


## 6. Training functions and parameter count
AdamW, clipping, BF16 when available (otherwise FP16 + scaling), token-based warmup/cosine schedule, periodic best/last checkpoints. Your model.pt export lacks the old optimizer; a fresh continuation starts a fresh optimizer; same-run resume restores optimizer/scaler/RNG. Trust only your own checkpoint files. Run one model per notebook. Stop the training cell with Interrupt, not kernel shutdown; abrupt shutdown resumes from the last periodic checkpoint.

In [ ]:
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
AMP_DTYPE=torch.bfloat16 if DEVICE=='cuda' and torch.cuda.is_bf16_supported() else torch.float16

def amp():return torch.autocast('cuda',dtype=AMP_DTYPE) if DEVICE=='cuda' else nullcontext()
def sync():
    if DEVICE=='cuda':torch.cuda.synchronize()
def atomic_torch(obj,path):
    p=Path(path);torch.save(obj,str(p)+'.tmp');os.replace(str(p)+'.tmp',p)

def make_model():return LegalLM(CFG) if ARCH=='lexweave' else BridgeLM(CFG)

def initial_model():
    m=make_model()
    if START_MODE=='continue':
        ck=torch.load(OLD_CHECKPOINT,map_location='cpu',weights_only=False)
        assert ck.get('tokenizer_sha',ck.get('train_config',{}).get('tokenizer_sha'))==sha(CACHE/'tokenizer.json'),'Tokenizer mismatch'
        if not isinstance(ck,dict) or 'model_config' not in ck or 'model' not in ck:
            raise ValueError('Unexpected model.pt layout; provide checkpoint keys for inspection.')
        old=ck['model_config']
        for k,v in CFG.items():
            if k not in ('grad_checkpoint',):assert old.get(k)==v,f'Checkpoint architecture mismatch: {k}'
        m.load_state_dict(ck['model'],strict=True)
        del ck
    return m

@torch.no_grad()
def validate(m):
    m.eval();rng=np.random.default_rng(SEED+9001);total=0.;nt=0
    for _ in range(EVAL_BATCHES):
        x,y,src=batch(VAL,rng,ARCH)
        with amp():loss=m(x,src=src,targets=y)
        n=int(y.ne(-100).sum());total+=float(loss)*n;nt+=n
    m.train();return total/max(1,nt)

def experiment_config():
    config=dict(model=CFG,arch=ARCH,seed=SEED,start_mode=START_MODE,
                initial_checkpoint_sha=sha(OLD_CHECKPOINT) if START_MODE=='continue' else None,
                tokenizer_sha=sha(CACHE/'tokenizer.json'),data_sha=sha(CACHE/'data_config.json'),
                packed_report_sha=sha(PACK/'report.json'),target_tokens=TRAIN_TARGET_TOKENS,
                micro=MICRO_BATCH,accum=ACCUM,lr=LR,min_lr=MIN_LR,warmup_tokens=WARMUP_TOKENS,
                weights=SOURCE_WEIGHTS,source_context=SOURCE_CONTEXT,target_context=TARGET_CONTEXT,
                eval_batches=EVAL_BATCHES)
    return json.loads(json.dumps(config))

def benchmark(updates=3):
    assert DEVICE=='cuda','Run preprocessing on CPU; switch to a GPU for this cell.'
    with torch.random.fork_rng(devices=[torch.cuda.current_device()]):
        m=make_model().to(DEVICE).train();opt=torch.optim.AdamW(m.parameters(),lr=LR,betas=(.9,.95),weight_decay=.1)
        scaler=torch.amp.GradScaler('cuda',enabled=AMP_DTYPE==torch.float16)
        rng=np.random.default_rng(SEED+123);dur=[];tokens=[]
        torch.cuda.reset_peak_memory_stats()
        for i in range(updates+1):
            sync();t=time.monotonic();opt.zero_grad(set_to_none=True);nt=0
            for _ in range(ACCUM):
                x,y,src=batch(TRAIN,rng,ARCH)
                with amp():loss=m(x,src=src,targets=y)/ACCUM
                scaler.scale(loss).backward();nt+=int(y.ne(-100).sum())
            scaler.unscale_(opt);nn.utils.clip_grad_norm_(m.parameters(),1.);scaler.step(opt);scaler.update();sync()
            if i:dur.append(time.monotonic()-t);tokens.append(nt)
        tps=sum(tokens)/sum(dur)
        result=dict(gpu=torch.cuda.get_device_name(),parameters=sum(p.numel() for p in m.parameters()),
                    seconds_per_update=float(np.mean(dur)),loss_tokens_per_second=tps,
                    estimated_training_hours=TRAIN_TARGET_TOKENS/tps/3600,
                    peak_allocated_GiB=torch.cuda.max_memory_allocated()/2**30,
                    note='Short benchmark; excludes validation, generation and checkpoints. One GPU only.')
        atomic_json(result,RUN/'benchmark.json');print(result)
        del m,opt,scaler;gc.collect();torch.cuda.empty_cache()
    return result

def train_model():
    assert DEVICE=='cuda','Enable a CUDA GPU.'
    random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED);torch.cuda.manual_seed_all(SEED)
    config=experiment_config();config_path=RUN/'experiment.json'
    if config_path.exists():assert json.loads(config_path.read_text())==config,'Config changed: restore settings or choose new RUN_TAG.'
    else:atomic_json(config,config_path)
    last=RUN/'last.pt'
    m=(make_model() if last.exists() else initial_model()).to(DEVICE)
    opt=torch.optim.AdamW(m.parameters(),lr=LR,betas=(.9,.95),weight_decay=.1)
    scaler=torch.amp.GradScaler('cuda',enabled=AMP_DTYPE==torch.float16)
    rng=np.random.default_rng(SEED);step=ntokens=0;best=float('inf');elapsed_before=0.
    if last.exists():
        ck=torch.load(last,map_location='cpu',weights_only=False)
        assert ck['experiment']==config
        m.load_state_dict(ck['model']);opt.load_state_dict(ck['optimizer']);scaler.load_state_dict(ck['scaler'])
        step,ntokens,best,elapsed_before=ck['step'],ck['target_tokens'],ck['best'],ck['elapsed']
        rng.bit_generator.state=ck['numpy_rng'];random.setstate(ck['python_rng']);torch.set_rng_state(ck['torch_rng'])
        torch.cuda.set_rng_state_all(ck['cuda_rng']);del ck
        print('Resumed',step,'updates;',ntokens,'loss tokens')
    start=time.monotonic();m.train();replay=None;phase="idle"
    def save():
        atomic_torch(dict(model_config=CFG,model=m.state_dict(),optimizer=opt.state_dict(),scaler=scaler.state_dict(),
            experiment=config,step=step,target_tokens=ntokens,best=best,elapsed=elapsed_before+time.monotonic()-start,
            numpy_rng=rng.bit_generator.state,python_rng=random.getstate(),torch_rng=torch.get_rng_state(),
            cuda_rng=torch.cuda.get_rng_state_all()),last)
    def assess():
        nonlocal best
        val=validate(m)
        if not math.isfinite(val):raise FloatingPointError('Nonfinite validation loss')
        row=dict(step=step,target_tokens=ntokens,val_loss=val,elapsed_seconds=elapsed_before+time.monotonic()-start)
        with (RUN/'history.jsonl').open('a') as f:f.write(json.dumps(row)+'\n')
        print(row,flush=True)
        if val<best:
            best=val;atomic_torch(dict(model_config=CFG,model=m.state_dict(),tokenizer_sha=sha(CACHE/'tokenizer.json'),
                step=step,target_tokens=ntokens,val_loss=val,experiment=config),RUN/'best.pt')
    try:
        while ntokens<TRAIN_TARGET_TOKENS and (time.monotonic()-start)/3600<MAX_HOURS:
            p=max(0.,min(1.,(ntokens-WARMUP_TOKENS)/max(1,TRAIN_TARGET_TOKENS-WARMUP_TOKENS)))
            lr=LR*(ntokens+1)/max(1,WARMUP_TOKENS) if ntokens<WARMUP_TOKENS else MIN_LR+.5*(LR-MIN_LR)*(1+math.cos(math.pi*p))
            for group in opt.param_groups:group['lr']=lr
            # Snapshot RNG before accumulation, so interrupted updates can be replayed exactly.
            phase="accumulation"
            replay=(json.loads(json.dumps(rng.bit_generator.state)),torch.get_rng_state(),torch.cuda.get_rng_state_all())
            opt.zero_grad(set_to_none=True);n=0;loss_sum=0
            for _ in range(ACCUM):
                x,y,src=batch(TRAIN,rng,ARCH)
                with amp():loss=m(x,src=src,targets=y)/ACCUM
                if not torch.isfinite(loss):raise FloatingPointError('Nonfinite loss')
                scaler.scale(loss).backward();n+=int(y.ne(-100).sum());loss_sum+=float(loss.detach())
            scaler.unscale_(opt);norm=nn.utils.clip_grad_norm_(m.parameters(),1.)
            if not torch.isfinite(norm):raise FloatingPointError('Nonfinite gradient; rerun with BF16 or investigate LR')
            phase="optimizer"
            scaler.step(opt);scaler.update();step+=1;ntokens+=n;replay=None;phase="idle"
            if step%20==0:print('update',step,'target tokens',ntokens,'loss',round(loss_sum,4),'lr',lr,flush=True)
            if step%EVAL_EVERY==0:assess()
            if step%SAVE_EVERY==0:save()
    except KeyboardInterrupt:
        if phase=="optimizer":
            raise RuntimeError("Interrupted during optimizer mutation; rerun from last.pt. No partial state saved.")
        if replay is not None:
            rng.bit_generator.state=replay[0];torch.set_rng_state(replay[1]);torch.cuda.set_rng_state_all(replay[2])
        print('Interrupted; saving completed updates. Rerun to resume.')
    except BaseException:
        print('Error: resume from the last committed checkpoint, not a partially updated optimizer.')
        raise
    else:
        if step:assess()
    save()
    del m,opt,scaler;gc.collect();torch.cuda.empty_cache()
    print('Run saved:',RUN)


In [ ]:
# Counts use the meta device, avoiding allocation of a second full-size model.
with torch.device('meta'):probe=make_model()
n=sum(p.numel() for p in probe.parameters());del probe
assert 50_000_000 <= n <= 250_000_000, f'{n:,} parameters exceeds course/user bounds'
print(ARCH,MODEL_SIZE,f'{n:,} parameters')
if DEVICE=='cuda':
    print(torch.cuda.get_device_name(), 'GPUs visible:',torch.cuda.device_count(), 'this notebook uses ONE GPU')
else:print('CPU preprocessing is supported; training requires a CUDA PyTorch kernel.')
# Save a standalone implementation/config next to the checkpoint for later phases.
atomic_json(CFG,RUN/'model_config.json')


## 7. Correctness smoke test and real GPU benchmark
Benchmark allocates a disposable model and optimizer. It does not train your run or overwrite checkpoints. Adjust MICRO_BATCH/ACCUM/GRAD_CHECKPOINT before training; later changes require a new RUN_TAG or the original settings to resume.

In [ ]:
# Tiny models test mechanics; these are NOT trained ablations or evidence of quality.
def smoke():
    global CFG
    original=CFG
    try:
        if ARCH=='lexweave':
            CFG=dict(original,dim=48,heads=6,kv_heads=2,layers=2,ff_dim=128,
                     mem_dim=24,mem_heads=3,conv_dim=24,vocab=128,context=96)
        else:
            CFG=dict(original,dim=48,heads=6,encoder_layers=2,decoder_layers=2,ff_dim=128,
                     latent_slots=4,vocab=128,context=96,source_context=96)
        torch.manual_seed(SEED);m=make_model().cpu();x=torch.randint(4,128,(2,70));src=torch.randint(4,128,(2,50))
        src[:,-5:]=PAD
        m.eval()
        with torch.no_grad():
            h=m.hidden(x,src=src if ARCH=='bridge' else None)
            changed=x.clone();changed[:,40:]=torch.randint(4,128,changed[:,40:].shape)
            h2=m.hidden(changed,src=src if ARCH=='bridge' else None)
            assert torch.allclose(h[:,:40],h2[:,:40],atol=2e-5), 'Future-token leak'
            hp=m.hidden(x[:,:40],src=src if ARCH=='bridge' else None)
            assert torch.allclose(h[:,:40],hp,atol=2e-5), 'Prefix mismatch'
            if ARCH=='bridge':
                compact=m.hidden(x,src=src[:,:45])
                assert torch.allclose(h,compact,atol=2e-5), 'Encoder padding leak'
        m.train();y=torch.randint(4,128,x.shape);y[:,-7:]=-100
        loss=m(x,src=src if ARCH=='bridge' else None,targets=y);loss.backward()
        assert torch.isfinite(loss) and all(torch.isfinite(p.grad).all() for p in m.parameters() if p.grad is not None)
        print('Causality, prefix consistency, masking and gradients passed; loss',float(loss.detach()))
    finally:CFG=original
smoke()


In [ ]:
benchmark_result=benchmark(updates=3)

## 8. Train this one model
This starts a paid GPU workload when you execute it. Inspect benchmark ETA first. EVAL_EVERY and SAVE_EVERY control overhead/recovery. The 1B loss-token budget may span multiple invocations. Resume by rerunning definitions/settings and this cell with the same experiment settings.

In [ ]:
train_model()

## 9. Best checkpoint and guarded generation
EOS is suppressed to meet the supplied fixed 128-word requirement; repetition controls can distort natural text. Tune on held-out continuations, never private references. Reaching 128 words is format validity, not proof of quality. No filler is fabricated. Generation has no KV cache and may take substantial time for 1,000 prompts.

In [ ]:
# Load exactly this run's best measured validation checkpoint.
best_path=RUN/'best.pt'
if not best_path.exists():
    raise FileNotFoundError('No best.pt yet: resume training until a validation completes.')
ck=torch.load(best_path,map_location='cpu',weights_only=False)
assert ck['model_config']==CFG
assert ck['tokenizer_sha']==sha(CACHE/'tokenizer.json')
model=make_model();model.load_state_dict(ck['model']);model=model.to(DEVICE).eval()
print('Loaded step',ck['step'],'loss tokens',ck['target_tokens'],'val loss',ck['val_loss'])
del ck

GEN_TEMPERATURE=.8
GEN_TOP_P=.9
GEN_REPETITION_PENALTY=1.15
MAX_NEW_TOKENS=2048
NO_REPEAT_NGRAM=4
PUNCT_REPEAT_LIMIT=6
DECODER_VERSION='sagemaker_v1'
punct=[]
for i in range(tok.get_vocab_size()):
    piece=tok.decode([i],skip_special_tokens=False)
    if piece and not any(c.isalnum() for c in piece):punct.append(i)
punct=torch.tensor(punct,device=DEVICE,dtype=torch.long)

@torch.inference_mode()
def continue_text(prompt,seed=42):
    model.eval();ids=tok.encode(prompt).ids or [BOS]
    if ARCH=='bridge':
        with amp():memory=model.encode(torch.tensor([ids[-SOURCE_CONTEXT:]],device=DEVICE))
        current=[BOS]
    else:memory=None;current=ids[-CONTEXT:]
    gen=[];ngrams=defaultdict(set);text='';interventions=0
    rng=torch.Generator(device=DEVICE).manual_seed(seed)
    for _ in range(MAX_NEW_TOKENS):
        x=torch.tensor([current[-CONTEXT:]],device=DEVICE)
        with amp():logits=model(x,memory=memory)[0].float().clone()
        if not torch.isfinite(logits).all():raise FloatingPointError('Nonfinite model logits')
        logits[[PAD,BOS,EOS,UNK]]=-float('inf')
        if gen:
            seen=list(set(gen[-256:]));z=logits[seen]
            logits[seen]=torch.where(z>=0,z/GEN_REPETITION_PENALTY,z*GEN_REPETITION_PENALTY)
        if len(gen)>=NO_REPEAT_NGRAM-1:
            banned=ngrams.get(tuple(gen[-(NO_REPEAT_NGRAM-1):]),())
            if banned:logits[list(banned)]=-float('inf')
        if re.search(r'([^\w\s])(?:\s*\1){5,}$',text) or len(text)-len(text.rstrip())>=16:
            logits[punct]=-float('inf');interventions+=1
        if not torch.isfinite(logits).any():raise RuntimeError('All tokens blocked; inspect decoding settings')
        if GEN_TEMPERATURE==0:nxt=int(logits.argmax())
        else:
            scores,order=torch.sort(logits/GEN_TEMPERATURE,descending=True);p=scores.softmax(-1)
            scores[(p.cumsum(-1)-p)>GEN_TOP_P]=-float('inf')
            nxt=int(order[torch.multinomial(scores.softmax(-1),1,generator=rng)])
        gen.append(nxt);current.append(nxt)
        if len(gen)>=NO_REPEAT_NGRAM:ngrams[tuple(gen[-NO_REPEAT_NGRAM:-1])].add(nxt)
        text=tok.decode(gen,skip_special_tokens=True)
        if len(text.split())>=129:
            return ' '.join(text.split()[:128]),{'tokens':len(gen),'punctuation_interventions':interventions}
    raise RuntimeError(f'Only {len(text.split())} words; no filler added. Tail: {text[-200:]!r}')

def rouge_l(a,b):
    a=a.lower().split();b=b.lower().split();dp=[0]*(len(b)+1)
    for x in a:
        old=dp[:]
        for j,y in enumerate(b,1):dp[j]=old[j-1]+1 if x==y else max(old[j],dp[j-1])
    return 2*dp[-1]/max(1,len(a)+len(b))

def evaluate_continuations(n=16):
    # Same saved prompt/reference examples for both architectures; no outside evaluator downloaded.
    rows=json.loads((CACHE/'continuation_dev.json').read_text())
    # Interleave sources so a small evaluation doesn't select Acts alone.
    groups=[[r for r in rows if r['source']==s] for s in SOURCES]
    rows=[g[j] for j in range(max(map(len,groups),default=0)) for g in groups if j<len(g)][:n]
    result=[]
    for i,r in enumerate(rows):
        try:
            text,info=continue_text(r['prompt'],SEED+i)
            result.append({**r,'prediction':text,'rouge_l':rouge_l(text,r['reference']),**info})
        except RuntimeError as e:result.append({**r,'error':str(e),'rouge_l':0.})
        print(i+1,'/',len(rows),'ROUGE-L',result[-1]['rouge_l'],flush=True)
    atomic_json(result,RUN/'continuation_eval.json')
    print('Mean local ROUGE-L:',np.mean([r['rouge_l'] for r in result]) if result else None)
    print('Not the competition BERTScore. Inspect text and failures before changing settings.')
    return result

def generate_submission():
    path=official_test_path();rows=[(key,text_of(row)) for key,row in records(path,require_id=True)]
    ids=[key for key,_ in rows]
    assert len(ids)==len(set(ids)) and len(ids)>0
    cfg=dict(checkpoint_sha=sha(best_path),test_sha=sha(path),tokenizer_sha=sha(CACHE/'tokenizer.json'),
             decoder=DECODER_VERSION,temp=GEN_TEMPERATURE,top_p=GEN_TOP_P,penalty=GEN_REPETITION_PENALTY,
             max_tokens=MAX_NEW_TOKENS,ngram=NO_REPEAT_NGRAM,punct=PUNCT_REPEAT_LIMIT,seed=SEED)
    tag=hashlib.sha256(json.dumps(cfg,sort_keys=True).encode()).hexdigest()[:12]
    dest=RUN/('submission_'+tag);dest.mkdir(exist_ok=True)
    atomic_json(cfg,dest/'manifest.json');pred_path=dest/'prediction.json'
    preds=json.loads(pred_path.read_text()) if pred_path.exists() else {}
    assert set(preds)<=set(ids)
    for i,(key,prompt) in enumerate(rows):
        if key in preds:
            assert set(preds[key])=={'text'} and len(preds[key]['text'].split())==128
            continue
        try:text,info=continue_text(prompt,SEED+i)
        except Exception as e:
            atomic_json({'id':key,'error':repr(e)},dest/'failure.json');raise
        preds[key]={'text':text};atomic_json(preds,pred_path)
        print(f'Test {i+1}/{len(rows)} saved; tokens={info["tokens"]}',flush=True)
    assert set(preds)==set(ids)
    assert all(set(v)=={'text'} and isinstance(v['text'],str) and len(v['text'].split())==128 for v in preds.values())
    archive=RUN/(RUN.name+'_'+tag+'.zip')
    with zipfile.ZipFile(str(archive)+'.tmp','w',zipfile.ZIP_DEFLATED) as z:z.write(pred_path,'prediction.json')
    os.replace(str(archive)+'.tmp',archive)
    with zipfile.ZipFile(archive) as z:
        assert z.namelist()==['prediction.json'];assert json.loads(z.read('prediction.json'))==preds
    print('Validated submission:',archive)
    return archive


## 10. Inspect held-out predictions
Starts with 16 source-interleaved examples. Increase to 128 for a stronger local comparison. ROUGE-L is a cheap diagnostic, not a substitute for the competition primary BERTScore. All candidates should use the same held-out examples/decoding when compared.

In [ ]:
dev_results=evaluate_continuations(n=16)
for row in dev_results[:2]:
    print("PROMPT:",row["prompt"][-400:])
    print("PREDICTION:",row.get("prediction",row.get("error")))
    print("REFERENCE:",row["reference"],"\n")

## 11. Generate the official submission
Uses every exact official test ID. Saves each example and skips validated existing predictions on rerun. Decoder/checkpoint changes use a different folder. ZIP contains ONLY root prediction.json. Upload this ZIP to Codabench, not weights. Public test content is never used in preprocessing.

In [ ]:
archive=generate_submission()
from IPython.display import FileLink,display
display(FileLink(str(archive)))

## 12. Save reproducibility artifacts
Keep best.pt, last.pt, tokenizer.json, model source/config, data report, experiment.json, history.jsonl and the shared held-out set. Model files are for your records/later phases; they are not included in prediction ZIP. Back up WORK to your own persistent storage. Stop the SageMaker compute when finished; notebook completion does not shut it down.

In [ ]:
MODEL_SOURCE = '"""LEXWeave: a project-specific experimental causal LM, not a pretrained model.\n\nParallel token attention, strictly completed-chunk memory, and routed dilated\nconvolution, combined by a token-dependent gate. See notebook for hypotheses,\nprior work, limitations, and controlled ablations. No claim of literature novelty.\n"""\nimport math\nimport torch\nfrom torch import nn\nimport torch.nn.functional as F\nfrom torch.utils.checkpoint import checkpoint\n\n\nclass RMSNorm(nn.Module):\n    def __init__(self, dim):\n        super().__init__()\n        self.weight = nn.Parameter(torch.ones(dim))\n\n    def forward(self, x):\n        z = x.float()\n        return (z * torch.rsqrt(z.square().mean(-1, keepdim=True) + 1e-6)).to(x.dtype) * self.weight\n\n\nclass Rotary(nn.Module):\n    def __init__(self, head_dim):\n        super().__init__()\n        assert head_dim % 2 == 0\n        inv = 1.0 / (10000 ** (torch.arange(0, head_dim, 2).float() / head_dim))\n        self.register_buffer(\'inv\', inv, persistent=False)\n\n    def forward(self, x, positions):\n        a = torch.outer(positions.float(), self.inv.float())\n        c = a.cos().to(x.dtype)[None, None]\n        s = a.sin().to(x.dtype)[None, None]\n        u, v = x[..., ::2], x[..., 1::2]\n        return torch.stack((u*c-v*s, u*s+v*c), -1).flatten(-2)\n\n\nclass Attention(nn.Module):\n    """Exact causal token attention is retained for names, numbers and citations."""\n    def __init__(self, dim, heads, kv_heads):\n        super().__init__()\n        assert dim % heads == 0 and heads % kv_heads == 0\n        self.h, self.k, self.dh = heads, kv_heads, dim // heads\n        self.q = nn.Linear(dim, dim, bias=False)\n        self.kv = nn.Linear(dim, 2 * kv_heads * self.dh, bias=False)\n        self.out = nn.Linear(dim, dim, bias=False)\n        self.rope = Rotary(self.dh)\n\n    def forward(self, x):\n        b, t, d = x.shape\n        q = self.q(x).view(b, t, self.h, self.dh).transpose(1, 2)\n        k, v = self.kv(x).view(b, t, 2, self.k, self.dh).unbind(2)\n        k, v = k.transpose(1, 2), v.transpose(1, 2)\n        pos = torch.arange(t, device=x.device)\n        q, k = self.rope(q, pos), self.rope(k, pos)\n        if self.h != self.k:\n            k = k.repeat_interleave(self.h // self.k, dim=1)\n            v = v.repeat_interleave(self.h // self.k, dim=1)\n        z = F.scaled_dot_product_attention(q, k, v, is_causal=True, dropout_p=0.)\n        return self.out(z.transpose(1, 2).contiguous().view(b, t, d))\n\n\nclass ChunkMemory(nn.Module):\n    """Learn a salience-weighted vector per chunk; read only COMPLETED past chunks.\n\n    Memory is recomputed inside the current context, not carried between batches.\n    A fixed zero slot makes the first chunk well-defined without all-masked rows.\n    """\n    def __init__(self, dim, mem_dim=128, heads=4, chunk=32, learned_pool=True):\n        super().__init__()\n        assert mem_dim % heads == 0\n        self.dim, self.md, self.h = dim, mem_dim, heads\n        self.dh, self.chunk = mem_dim // heads, chunk\n        self.score = nn.Linear(dim, 1, bias=False) if learned_pool else None\n        self.q = nn.Linear(dim, mem_dim, bias=False)\n        self.kv = nn.Linear(dim, 2*mem_dim, bias=False)\n        self.out = nn.Linear(mem_dim, dim, bias=False)\n        self.rope = Rotary(self.dh)\n\n    def pool(self, x):\n        b, t, d = x.shape\n        c = self.chunk\n        n = (t+c-1)//c\n        padded = F.pad(x, (0, 0, 0, n*c-t)).view(b, n, c, d)\n        valid = torch.arange(n*c, device=x.device).view(1, n, c) < t\n        if self.score is None:\n            w = valid.float().expand(b, -1, -1)\n            w = w / w.sum(-1, keepdim=True).clamp_min(1)\n        else:\n            logits = self.score(padded).squeeze(-1).float()\n            logits = logits.masked_fill(~valid, -float(\'inf\'))\n            w = logits.softmax(-1)\n        summaries = (padded * w.to(padded.dtype).unsqueeze(-1)).sum(-2)\n        ends = (torch.arange(n, device=x.device)+1)*c - 1\n        return summaries, ends\n\n    def forward(self, x):\n        b, t, _ = x.shape\n        summary, ends = self.pool(x)\n        n = summary.shape[1]\n        q = self.q(x).view(b, t, self.h, self.dh).transpose(1, 2)\n        k, v = self.kv(summary).view(b, n, 2, self.h, self.dh).unbind(2)\n        k, v = k.transpose(1, 2), v.transpose(1, 2)\n        pos = torch.arange(t, device=x.device)\n        q, k = self.rope(q, pos), self.rope(k, ends)\n        # Null slot: exactly zero, no learned information from any future token.\n        k = torch.cat((k.new_zeros(b, self.h, 1, self.dh), k), dim=2)\n        v = torch.cat((v.new_zeros(b, self.h, 1, self.dh), v), dim=2)\n        allowed = ends[None, :] < pos[:, None]\n        allowed = torch.cat((torch.ones(t, 1, dtype=torch.bool, device=x.device), allowed), -1)\n        z = F.scaled_dot_product_attention(q, k, v, attn_mask=allowed[None, None], dropout_p=0.)\n        return self.out(z.transpose(1, 2).contiguous().view(b, t, self.md))\n\n\nclass PhraseMixer(nn.Module):\n    """Token-gated, left-padded depthwise convolutions at multiple spacings."""\n    def __init__(self, dim, inner=128, dilations=(1, 4, 16)):\n        super().__init__()\n        self.dilations = tuple(dilations)\n        self.up = nn.Linear(dim, inner, bias=False)\n        self.conv = nn.ModuleList([nn.Conv1d(inner, inner, 3, dilation=r,\n                                            groups=inner, bias=False) for r in self.dilations])\n        self.scale_gate = nn.Linear(inner, len(self.dilations), bias=False) if len(self.dilations)>1 else None\n        self.out = nn.Linear(inner, dim, bias=False)\n\n    def forward(self, x):\n        z = F.silu(self.up(x))\n        gate = self.scale_gate(z).float().softmax(-1).to(z.dtype) if self.scale_gate is not None else None\n        y = torch.zeros_like(z)\n        for i, (conv, r) in enumerate(zip(self.conv, self.dilations)):\n            branch = conv(F.pad(z.transpose(1, 2), (2*r, 0))).transpose(1, 2)\n            y = y + (branch * gate[..., i:i+1] if gate is not None else branch)\n        return self.out(F.silu(y))\n\n\nclass WeaveBlock(nn.Module):\n    def __init__(self, cfg):\n        super().__init__()\n        d, a = cfg[\'dim\'], cfg[\'ablation\']\n        self.n1, self.n2 = RMSNorm(d), RMSNorm(d)\n        self.attn = Attention(d, cfg[\'heads\'], cfg[\'kv_heads\'])\n        self.memory = None if a in (\'no_memory\', \'plain\') else ChunkMemory(\n            d, cfg[\'mem_dim\'], cfg[\'mem_heads\'], cfg[\'chunk\'], learned_pool=a != \'mean_pool\')\n        self.phrase = None if a in (\'no_conv\', \'plain\') else PhraseMixer(\n            d, cfg[\'conv_dim\'], (1,) if a == \'single_scale\' else cfg[\'dilations\'])\n        self.active = [0] + ([1] if self.memory is not None else []) + ([2] if self.phrase is not None else [])\n        self.router = nn.Linear(d, 3, bias=False) if a not in (\'fixed_mix\', \'plain\') else None\n        self.register_buffer(\'route_prior\', torch.tensor([2., 0., 0.]), persistent=True)\n        self.chunk = cfg[\'chunk\']\n        self.up = nn.Linear(d, 2*cfg[\'ff_dim\'], bias=False)\n        self.down = nn.Linear(cfg[\'ff_dim\'], d, bias=False)\n        self.record_routes = False\n        self.last_routes = None\n\n    def route_weights(self, x):\n        b, t, _ = x.shape\n        logits = self.route_prior[None, None].expand(b, t, 3)\n        if self.router is not None: logits = logits + self.router(x).float()\n        enabled = torch.zeros(t, 3, dtype=torch.bool, device=x.device)\n        enabled[:, self.active] = True\n        # Chunk memory carries no past summary during the first chunk.\n        enabled[:, 1] &= torch.arange(t, device=x.device) >= self.chunk\n        return logits.masked_fill(~enabled[None], -float(\'inf\')).softmax(-1).to(x.dtype)\n\n    def forward(self, x):\n        z = self.n1(x)\n        weights = self.route_weights(z)\n        mixed = weights[..., 0:1] * self.attn(z)\n        if self.memory is not None: mixed = mixed + weights[..., 1:2] * self.memory(z)\n        if self.phrase is not None: mixed = mixed + weights[..., 2:3] * self.phrase(z)\n        if self.record_routes and not self.training:\n            self.last_routes = weights.detach().float().mean((0, 1)).cpu()\n        x = x + mixed\n        a, b = self.up(self.n2(x)).chunk(2, -1)\n        return x + self.down(F.silu(a) * b)\n\n\nclass LegalLM(nn.Module):\n    def __init__(self, cfg):\n        super().__init__()\n        self.cfg, self.kind = dict(cfg), \'lexweave\'\n        self.grad_ckpt = cfg.get(\'grad_checkpoint\', True)\n        self.emb = nn.Embedding(cfg[\'vocab\'], cfg[\'dim\'])\n        self.blocks = nn.ModuleList([WeaveBlock(cfg) for _ in range(cfg[\'layers\'])])\n        self.norm = RMSNorm(cfg[\'dim\'])\n        self.apply(self.init_weights)\n        # Mild residual initialization keeps the extra paths from overwhelming input.\n        residual_std = 0.02 / math.sqrt(2*cfg[\'layers\'])\n        for block in self.blocks:\n            nn.init.normal_(block.attn.out.weight, std=residual_std)\n            nn.init.normal_(block.down.weight, std=residual_std)\n            if block.memory is not None:\n                nn.init.normal_(block.memory.out.weight, std=residual_std)\n                if block.memory.score is not None: nn.init.zeros_(block.memory.score.weight)\n            if block.phrase is not None:\n                nn.init.normal_(block.phrase.out.weight, std=residual_std)\n                if block.phrase.scale_gate is not None: nn.init.zeros_(block.phrase.scale_gate.weight)\n            if block.router is not None: nn.init.zeros_(block.router.weight)\n\n    @staticmethod\n    def init_weights(m):\n        if isinstance(m, (nn.Linear, nn.Embedding, nn.Conv1d)):\n            nn.init.normal_(m.weight, std=0.02)\n            if getattr(m, \'bias\', None) is not None: nn.init.zeros_(m.bias)\n\n    def hidden(self, ids, src=None, memory=None):\n        if ids.shape[1] > self.cfg[\'context\']: raise ValueError(\'Input exceeds configured context.\')\n        x = self.emb(ids)\n        for block in self.blocks:\n            x = checkpoint(block, x, use_reentrant=False) if self.training and self.grad_ckpt else block(x)\n        return self.norm(x)\n\n    def forward(self, ids, src=None, targets=None, memory=None):\n        x = self.hidden(ids)\n        if targets is None: return F.linear(x[:, -1], self.emb.weight).float()\n        def ce(h, y):\n            logits = F.linear(h, self.emb.weight).float()\n            return F.cross_entropy(logits.reshape(-1, self.cfg[\'vocab\']), y.reshape(-1), reduction=\'sum\')\n        total = x.new_zeros((), dtype=torch.float32)\n        for j in range(0, x.shape[1], 128):\n            h, y = x[:, j:j+128], targets[:, j:j+128]\n            total = total + (checkpoint(ce, h, y, use_reentrant=False) if self.training and self.grad_ckpt else ce(h, y))\n        return total / targets.ne(-100).sum().clamp_min(1)\n\n\n'
(RUN/"modeling.py").write_text(MODEL_SOURCE,encoding="utf-8")
shutil.copy2(CACHE/"tokenizer.json",RUN/"tokenizer.json")
shutil.copy2(CACHE/"data_report.json",RUN/"data_report.json")
atomic_json({"torch":torch.__version__,"cuda":torch.version.cuda,"architecture":ARCH},RUN/"environment.json")
print("Preserve run directory:",RUN)